In [1]:
!pip install -q xgboost

In [2]:
from google.colab import files

uploaded = files.upload()


Saving placement_predict_50k Dataset.csv to placement_predict_50k Dataset.csv


In [3]:
import time
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

from xgboost import XGBClassifier

In [5]:
df = pd.read_csv("placement_predict_50k Dataset.csv")

print("Dataset shape:", df.shape)
print("\nFirst 5 rows:")
display(df.head())

Dataset shape: (50000, 32)

First 5 rows:


,StudentID,Gender,City,CollegeTier,Stream,Specialisation,Hostel,HistoryOfBacklogs,SGPA_Sem1,SGPA_Sem2,...,Publications,AptitudeTestScore,SoftSkillsRating,CodingTestScore,MockInterviewScore,ExtraCurricular,CGPA_Tier,PlacementStatus,IsAnomaly,Salary Package
0,1,Male,Ahmedabad,Tier2,ECE,Networking,No,No,6.02,6.54,...,0,66.7,2.2,49.4,47.8,0,Low,0,0,0.00
1,2,Female,Mumbai,Tier2,ECE,DataScience,Yes,Yes,5.84,5.12,...,0,48.2,2.4,26.7,25.8,0,Low,0,0,0.00
2,3,Male,Kolkata,Tier2,IT,DataScience,Yes,No,4.91,5.29,...,0,73.8,2.8,67.7,41.5,0,Low,1,0,3.89
3,4,Male,Jaipur,Tier1,CS,AI,No,No,7.67,8.03,...,0,69.8,2.7,66.9,48.0,0,Mid,1,0,8.37
4,5,Male,Pune,Tier2,IT,DataScience,Yes,No,8.14,8.97,...,1,73.1,2.1,71.7,61.7,1,High,1,0,18.99


In [6]:
# Remove anomaly column if present
if "IsAnomaly" in df.columns:
    df = df.drop(columns=["IsAnomaly"])

# Target
y = df["PlacementStatus"].astype(int)

# Features
X = df.drop(columns=["PlacementStatus"])

# Find categorical and numerical columns
cat_cols = X.select_dtypes(exclude="number").columns.tolist()
num_cols = X.select_dtypes(include="number").columns.tolist()

# Encode categorical columns
encoders = {}

for c in cat_cols:
    le = LabelEncoder()
    X[c] = le.fit_transform(X[c].astype(str))
    encoders[c] = le

# Fill missing numerical values
imputer = SimpleImputer(strategy="median")
X[num_cols] = imputer.fit_transform(X[num_cols])

# Scale numerical columns
scaler = StandardScaler()
X[num_cols] = scaler.fit_transform(X[num_cols])

print("Preprocessing completed.")
print("Features:", X.shape)
print("Target:", y.shape)

Preprocessing completed.
Features: (50000, 30)
Target: (50000,)


In [7]:
RANDOM_STATE = 42

VAL_SIZE = 0.15
TEST_SIZE = 0.15

# First split test data
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE
)

# Then split validation data
val_ratio = VAL_SIZE / (1 - TEST_SIZE)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val,
    y_train_val,
    test_size=val_ratio,
    stratify=y_train_val,
    random_state=RANDOM_STATE
)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

Train: (34999, 30)
Validation: (7501, 30)
Test: (7500, 30)


In [8]:
ada_base = DecisionTreeClassifier(
    max_depth=2,
    random_state=RANDOM_STATE
)

ada = AdaBoostClassifier(
    estimator=ada_base,
    n_estimators=200,
    learning_rate=0.5,
    random_state=RANDOM_STATE
)

start_time = time.time()

ada.fit(X_train, y_train)

ada_time = time.time() - start_time

print("AdaBoost training completed.")
print("Training time:", round(ada_time, 2), "seconds")

AdaBoost training completed.
Training time: 37.14 seconds


In [9]:
ada_pred = ada.predict(X_val)
ada_proba = ada.predict_proba(X_val)[:, 1]

ada_accuracy = accuracy_score(y_val, ada_pred)
ada_f1 = f1_score(y_val, ada_pred)
ada_auc = roc_auc_score(y_val, ada_proba)

print("AdaBoost Results")
print("----------------")
print("Accuracy :", round(ada_accuracy, 4))
print("F1 Score :", round(ada_f1, 4))
print("ROC-AUC  :", round(ada_auc, 4))

AdaBoost Results
----------------
Accuracy : 0.9984
F1 Score : 0.9988
ROC-AUC  : 0.9999


In [10]:
xgb = XGBClassifier(
    n_estimators=1000,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    early_stopping_rounds=30,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

start_time = time.time()

xgb.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)

xgb_time = time.time() - start_time

print("XGBoost training completed.")
print("Training time:", round(xgb_time, 2), "seconds")

XGBoost training completed.
Training time: 2.87 seconds


In [11]:
xgb_pred = xgb.predict(X_val)
xgb_proba = xgb.predict_proba(X_val)[:, 1]

xgb_accuracy = accuracy_score(y_val, xgb_pred)
xgb_f1 = f1_score(y_val, xgb_pred)
xgb_auc = roc_auc_score(y_val, xgb_proba)

print("XGBoost Results")
print("----------------")
print("Accuracy :", round(xgb_accuracy, 4))
print("F1 Score :", round(xgb_f1, 4))
print("ROC-AUC  :", round(xgb_auc, 4))
print("Best number of estimators:", xgb.best_iteration + 1)

XGBoost Results
----------------
Accuracy : 0.9989
F1 Score : 0.9992
ROC-AUC  : 0.9999
Best number of estimators: 174


In [12]:
results = pd.DataFrame({
    "Model": ["AdaBoost", "XGBoost"],
    "Accuracy": [ada_accuracy, xgb_accuracy],
    "F1 Score": [ada_f1, xgb_f1],
    "ROC-AUC": [ada_auc, xgb_auc],
    "Training Time (sec)": [ada_time, xgb_time],
    "Estimators": [
        ada.n_estimators,
        xgb.best_iteration + 1
    ]
})

results = results.sort_values(
    by="Accuracy",
    ascending=False
).reset_index(drop=True)

display(results)

,Model,Accuracy,F1 Score,ROC-AUC,Training Time (sec),Estimators
0,XGBoost,0.998933,0.999189,0.999929,2.873814,174
1,AdaBoost,0.998400,0.998783,0.999914,37.144700,200


In [13]:
results.to_csv("boosting_benchmark_results.csv", index=False)

print("Results saved successfully.")

Results saved successfully.


In [14]:
from google.colab import files

files.download("boosting_benchmark_results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>